# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}\nVersion: {metadata.version}\nLicense: {metadata.license}")
print(f"Keywords: {', '.join(metadata.keywords)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The FAIR² dataset groups tabular clinical records into record sets, each representing a different table. Each record set, field, and column is referenced by its unique `@id`.

In [ ]:
# Get available record sets from dataset.metadata
record_sets = metadata.recordSet

print("Available Record Sets and their @id:")
record_set_ids = []
for rs in record_sets:
    print(f"Name: {getattr(rs, 'name', 'N/A')}, Description: {getattr(rs, 'description', '')}")
    print(f"@id: {rs['@id']}\nFields:")
    record_set_ids.append(rs['@id'])
    for field in rs.get('field', []):
        print(f"  - {field['@id']} : {field['name']}")
    print()
# For demonstration, print a preview of records in the first record set
if record_sets:
    preview_rs_id = record_sets[0]['@id']
    for x in dataset.records(record_set=preview_rs_id):
        print(x)
        # Only show 1 preview
        break

## 3. Data Extraction
Load data from record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
# Use @id references dynamically from previous overview
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nRecord Set @id: {record_set_id}")
    print("Columns (referenced by field @id):")
    print(df.columns.tolist())
    print(df.head(2))
# Choose one for demonstration (Table 1 clinical features)
main_record_set_id = record_set_ids[0]
print("\nSelected record set (Table 1) preview:")
print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, grouping.
All column selections use their field `@id` values.

In [ ]:
# Example: Analyze "age at diagnosis" for the first record set.
# Find the @id of the 'Age at Diagnosis' field
clinical_rs = [rs for rs in record_sets if rs['@id'] == main_record_set_id][0]
age_field_id = None
for field in clinical_rs.get('field', []):
    if 'age' in field['name'].lower():
        age_field_id = field['@id']
        break
if age_field_id is None:
    raise ValueError("Could not find 'age at diagnosis' field in Table 1.")
# Assume the field is numeric
df = dataframes[main_record_set_id]
numeric_field = age_field_id
threshold = 20
if numeric_field in df.columns:
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize age
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Try grouping by 'Presence of Oligomenorrhea' (if present)
    group_field_id = None
    for field in clinical_rs.get('field', []):
        if 'oligomenorrhea' in field['name'].lower():
            group_field_id = field['@id']
            break
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field_id} (Presence of Oligomenorrhea):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

### Example: Visualize Age Distribution and its relation to Oligomenorrhea

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot Age at Diagnosis distribution
plt.figure(figsize=(6,4))
sns.histplot(df[numeric_field].dropna(), bins=10, kde=True)
plt.title('Age at Diagnosis Distribution')
plt.xlabel('Age at Diagnosis')
plt.ylabel('Frequency')
plt.show()

# Plot relation between Age and Oligomenorrhea (if present)
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(6,4))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field])
    plt.title('Age at Diagnosis vs. Oligomenorrhea')
    plt.xlabel('Oligomenorrhea Present')
    plt.ylabel('Age at Diagnosis')
    plt.show()

## 6. Conclusion
Key findings:
- Dataset loaded, demonstrating record set and field extraction by `@id`.
- Explored the clinical features table, focusing on age at diagnosis.
- Applied filtering and normalization to numeric fields, grouping by categorical attributes referenced by `@id`.
- Visualized age distributions and binary clinical features.

The FAIR² dataset provides detailed genotype–phenotype association for three patients. While small sample size limits inference, the structure supports reproducible data exploration using Croissant-compliant tools.